In [1]:
import pandas as pd
import nlp.config_cleaner as config

## `import pandas as pd`

**Purpose:**  
Bring in the Pandas library under the short name `pd`.

**Behavior:**
- Makes functions like `pd.read_csv()` and the `DataFrame` type available.
- Provides tools for loading, filtering, and saving table-shaped data.

**Effect:**  
Enables the program to work with CSV files as structured tables instead of raw text.

**Plain meaning:**  
Give the program the ability to read and manipulate spreadsheets.

---

## `import nlp.config_cleaner as config`

**Purpose:**  
Import the project’s cleaning configuration under the name `config`.

**Behavior:**
- Loads constants such as `KEEP_COLUMNS`, `FILTER_ROW`, `DEFAULT_DATAFILE`, and output settings.
- Allows access using `config.KEEP_COLUMNS`, `config.OUTPUT_PATH`, etc.

**Effect:**  
Separates “what the program does” from “how it is configured”.

**Plain meaning:**  
Load all the cleaning rules and file settings from one central place.


# Class: `CsvCommentCleaner`  

A class is a reusable tool that stores your data and provides actions you can run on it.  
Every function in this file is technically in the class `CsvCommentCleaner`

In [2]:
def __init__(self) -> None:
        self._validate_config()

## `__init__(self) -> None`

**Purpose:**  
Create a new cleaner object and immediately confirm the configuration is usable.

**Behavior:**
- Calls `_validate_config()` as soon as the object is created.

**Effect:**  
Stops the program early if the settings are wrong, instead of failing later in the middle of processing.

**Plain meaning:**  
Before doing any work, make sure the “settings file” is not broken.


In [3]:
def _validate_config(self) -> None:
        if not config.KEEP_COLUMNS or len(config.KEEP_COLUMNS) < 2:
            raise ValueError("KEEP_COLUMNS must contain at least two column names.")
        if not isinstance(config.FILTER_ROW, str) or config.FILTER_ROW == "":
            raise ValueError("FILTER_ROW must be a non-empty string.")
        if not isinstance(config.DEFAULT_DATAFILE, str) or config.DEFAULT_DATAFILE == "":
            raise ValueError("DEFAULT_DATAFILE must be a non-empty string.")

## `_validate_config(self) -> None`

**Purpose:**  
Check that required configuration values exist and have valid types.

**Behavior:**
- Confirms `config.KEEP_COLUMNS` exists and has at least two column names.
- Confirms `config.FILTER_ROW` is a non-empty string.
- Confirms `config.DEFAULT_DATAFILE` is a non-empty string.
- Raises `ValueError` with a clear message if any check fails.

**Effect:**  
Guarantees the rest of the class can rely on the config values being present and meaningful.

**Plain meaning:**  
Make sure the program has the minimum information needed to clean the CSV.


In [4]:
@staticmethod
def _is_all_non_ascii(val: object) -> bool:
    if not isinstance(val, str) or val == "":
        return False
    return all(ord(ch) > 127 for ch in val)

## `_is_all_non_ascii(val: object) -> bool`

**Purpose:**  
Detect strings made entirely of non-ASCII characters.

**Behavior:**
- If `val` is not a string or is empty, returns `False`.
- Otherwise checks every character and returns `True` only if all characters have a code point greater than 127.

**Effect:**  
Provides a filter rule to remove rows that contain text that is fully “non-standard ASCII” (depending on your data, this may catch rows in other languages or corrupted text).

**Plain meaning:**  
Return `True` if the text looks like it is made of only “non-English / non-standard” characters.


In [5]:
def load(self) -> pd.DataFrame:
        return pd.read_csv(config.DEFAULT_DATAFILE)

## `load(self) -> pd.DataFrame`

**Purpose:**  
Load the CSV file into a Pandas DataFrame.

**Behavior:**
- Reads the file path stored in `config.DEFAULT_DATAFILE` using `pd.read_csv(...)`.

**Effect:**  
Turns the CSV into a table object the rest of the code can filter.

**Plain meaning:**  
Open the CSV file and turn it into a table.

In [6]:
def filter_comments(self, df: pd.DataFrame) -> pd.DataFrame:
        keep_cols = config.KEEP_COLUMNS
        filter_row = config.FILTER_ROW

        missing = [c for c in keep_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Missing required column(s): {missing}")

        df = df[keep_cols]
        df = df[df[keep_cols[0]].str.contains(filter_row, case=False, na=False)]

        non_ascii_mask = df[keep_cols[1]].apply(self._is_all_non_ascii)
        df = df[~non_ascii_mask]

        return df

## `filter_comments(self, df: pd.DataFrame) -> pd.DataFrame`

**Purpose:**  
Keep only certain columns, keep only certain rows, and remove rows with unwanted text patterns.

**Behavior:**
- Reads `KEEP_COLUMNS` and `FILTER_ROW` from the config.
- Checks that all required columns exist in the DataFrame; if any are missing, raises `KeyError`.
- Reduces the DataFrame to only the `KEEP_COLUMNS`.
- Filters rows to only those where the first kept column contains `FILTER_ROW` (case-insensitive), treating missing values as non-matches.
- Builds a mask for rows where the second kept column is entirely non-ASCII using `_is_all_non_ascii`.
- Removes rows matching that non-ASCII mask.

**Effect:**  
Returns a smaller, cleaner table that matches the project’s filtering rules.

**Plain meaning:**  
Keep the columns you care about, keep rows that match a keyword, and remove rows that look like garbage text.


In [7]:
def run(self) -> pd.DataFrame:
        df = self.load()
        return self.filter_comments(df)

## `run(self) -> pd.DataFrame`

**Purpose:**  
Perform the complete cleaning operation in one call.

**Behavior:**
- Calls `load()` to read the file.
- Passes the result into `filter_comments()`.

**Effect:**  
One function that does the entire “load then clean” workflow.

**Plain meaning:**  
Do the whole job and return the cleaned table.


In [8]:
def save(self, df: pd.DataFrame) -> None:
        df.to_csv(
            config.OUTPUT_PATH,
            index=config.OUTPUT_INDEX,
            encoding=config.OUTPUT_ENCODING,
            sep=config.OUTPUT_SEPARATOR,
        )
    print("Saved to: " + config.OUTPUT_PATH)

## `save(self, df: pd.DataFrame) -> None`

**Purpose:**  
Write the cleaned DataFrame back out to a CSV file.

**Behavior:**
- Saves to `config.OUTPUT_PATH`.
- Uses config values for:
  - whether to include the index (`OUTPUT_INDEX`)
  - output encoding (`OUTPUT_ENCODING`)
  - separator character (`OUTPUT_SEPARATOR`)

**Effect:**  
Creates an output file in the exact format the config specifies.

**Plain meaning:**  
Save the cleaned results to a CSV, using the output settings.


In [9]:
# if __name == "__main__":
from nlp.SocialCleaner import CsvCommentCleaner
cleaner = CsvCommentCleaner()
comments_df = cleaner.run() 
comments_df.head()
cleaner.save(comments_df)

      Content type                                            Content
12         Comment                                     Loooks great .
14         Comment                                        Where at??)
17         Comment      I still want that big clock on the wall lol 😊
63         Comment                    Hello Do you need an employee ?
67         Comment                                               Nice
...            ...                                                ...
21710      Comment  @JasonNixonAB @mustardseedcan \nNot surprised....
21739      Comment  Hey man, that's awesome.I need people like you...
21740      Comment  Good services to building our communities. Tha...
21748      Comment  This the mustard seed in Calgary? My dad used ...
21761      Comment  I would include toothpaste, toothbrush, socks,...

[2093 rows x 2 columns]


## Execution walkthrough

This section describes the actual runtime flow. Please refer to the above cells as a glossary.

1. Python starts at `if __name__ == "__main__":`.

2. `cleaner = CsvCommentCleaner()` runs, which triggers `CsvCommentCleaner.__init__`.

3. Inside `CsvCommentCleaner.__init__`, `_validate_config` is called to confirm required config values are present and valid.

4. `comments_df = cleaner.run()` runs, which calls `CsvCommentCleaner.run`.

5. Inside `CsvCommentCleaner.run`, `load` is called to read `config.DEFAULT_DATAFILE` into a DataFrame.

6. Still inside `CsvCommentCleaner.run`, `filter_comments` is called with the loaded DataFrame.

7. Inside `CsvCommentCleaner.filter_comments`, the program checks that every column listed in `config.KEEP_COLUMNS` actually exists in the CSV; if not, it raises a `KeyError` and the program stops.

8. If the required columns exist, the DataFrame is reduced to only the columns in `config.KEEP_COLUMNS`.

9. The DataFrame is then filtered down to only rows where the first kept column contains `config.FILTER_ROW` (case-insensitive).

10. A second filter is applied: for each row, `_is_all_non_ascii` checks whether the text in the second kept column is made entirely of non-ASCII characters.

11. Any rows that `_is_all_non_ascii` flags as “all non-ASCII” are removed from the DataFrame.

12. `filter_comments` returns the cleaned DataFrame back to `run`.

13. `run` returns the cleaned DataFrame back to the main block as `comments_df`.

14. `print(comments_df.head())` prints the first few rows so you can verify the output.

15. `cleaner.save(comments_df)` runs, which calls `CsvCommentCleaner.save`.

16. Inside `CsvCommentCleaner.save`, the cleaned DataFrame is written to disk using the output settings in `config` (path, encoding, separator, and whether to include the index).

Result:  
If configuration and CSV inputs are valid, the program produces a cleaned DataFrame, prints a preview, and writes a cleaned CSV file.


## Runtime story  

When the file is executed directly, Python enters at the `if __name__ == "__main__":` block. The first thing that happens is the creation of a `CsvCommentCleaner` object. This immediately invokes `__init__`, which in turn calls `_validate_config`. Before any file is touched or any data is processed, the program confirms that the configuration makes sense: there must be at least two columns to keep, the filter string must exist, and a default data file must be defined. If any of these are wrong, the program stops here with a clear error, preventing confusing failures later.

Assuming the configuration is valid, the script asks the cleaner to “run.” This is where the actual work begins. The `run` method first loads the CSV file by calling `load`, which reads `config.DEFAULT_DATAFILE` into a Pandas DataFrame. At this point, the raw data exists in memory exactly as it appears in the file.

That raw table is then handed to `filter_comments`. This method becomes the gatekeeper of the dataset. It first checks that every column listed in `config.KEEP_COLUMNS` actually exists in the CSV. If even one is missing, the program halts with a precise error explaining what is wrong.

Once the structure is confirmed, the table is reduced to only the columns that matter. Everything else is discarded. The rows are then filtered so that only those whose first kept column contains the configured keyword remain. This step narrows the data to only the entries that are relevant to the task.

After that, each remaining row is inspected more closely. The second kept column is examined using `_is_all_non_ascii`. Any row whose text consists entirely of non-ASCII characters is removed. This quietly strips out rows that look like corrupted or unusable text.

The cleaned DataFrame flows back out of `filter_comments`, then out of `run`, and finally returns to the main block as `comments_df`. The script prints the first few rows so you can immediately see what survived the process.

Finally, the cleaned data is passed to `save`. This writes the result to disk using the output settings defined in the configuration: the file path, encoding, separator, and whether to include the index. At this point, the transformation is complete. The original CSV has been filtered, cleaned, previewed, and persisted as a new, controlled output file.


## Example
We begin with a file named `data.csv`.  
It contains three rows and the following columns:

`"Network","Channel name","Author name","Falcon user name","Content type","Content","Date created (UTC)"`

The rows look like this:

- A direct message asking about a green Lululemon hoodie  
- A Facebook post from “The Mustard Seed Thrift Store Edmonton”  
- A comment saying “Loooks great .”

At this point, nothing has been filtered or cleaned. The file is exactly as exported.

---

When the program is run, Python enters the main block and creates a `CsvCommentCleaner` object. As soon as the object exists, it checks the configuration. It verifies that the list of columns to keep is defined, that a filter keyword exists, and that a default CSV filename is set. Since the configuration is valid, execution continues.

The cleaner is then told to run. Internally, it starts by loading the CSV file. Pandas reads `data.csv` and converts it into a table in memory where each row corresponds to one of the three Facebook entries and each column corresponds to the CSV headers.

With the raw table loaded, the cleaner moves on to filtering. First, it checks that all required columns listed in `KEEP_COLUMNS` actually exist in the file. Because the CSV contains columns like `"Author name"` and `"Content"`, and those match the configuration, the program proceeds.

Next, the table is reduced to only the columns the cleaner cares about. Everything else — such as network name or timestamps — is dropped. The data is now smaller and focused.

The cleaner then applies its first content-based rule. It looks at the first kept column (for example, the author or organization name) and checks whether it contains the configured filter string. Rows that do not contain this keyword are removed. After this step, only rows that are relevant to the target organization remain.

Now the cleaner performs a final quality check. For each remaining row, it examines the text in the second kept column, usually the comment or message content. It checks whether that text consists entirely of non-ASCII characters. In this example, all remaining comments use normal English characters, so none of the rows are removed at this stage.

At the end of filtering, the cleaner returns a new table that contains only relevant rows, only relevant columns, and only usable text. This cleaned table is returned to the main block.

The program prints the first few rows so you can visually confirm the result. You see only the rows that matched the filter and passed the text checks.

Finally, the cleaner saves the cleaned table to a new CSV file using the output settings from the configuration. The file is written with the chosen encoding, separator, and index rules.

At this point, the process is complete. The original `data.csv` remains unchanged, and a cleaned version — smaller, filtered, and consistent — has been produced.
